# Customer Uplift & Causal Machine Learning Pipeline
### Double Machine Learning | Meta-Learners (S/T-Learners) | Doubly Robust AIPW | Criteo AI Uplift Benchmark

This computational engine estimates **Individual Treatment Effects (ITE)** and **Conditional Average Treatment Effects (CATE)**:
1. **Causal Problem Framing:** Standard ML predicts $P(Y=1 \mid X)$ (who buys), but Uplift ML predicts $\tau(X) = \mathbb{E}[Y(1) - Y(0) \mid X]$ (who buys *because* of marketing outreach).
2. **100,000-Record Criteo Benchmark:** Ingesting randomized controlled trial (RCT) data with 85% treated / 15% control distribution across 12 anonymized behavioral signals.
3. **Causal Architectures:** Training Single-Model S-Learner, Two-Model T-Learner, and Chernozhukov 5-Fold Doubly Robust AIPW (DML).
4. **Evaluation:** Assessing population-adjusted Qini Curves and Area Under the Uplift Curve (AUUC).

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Add root directory to path
sys.path.insert(0, os.getcwd())

from src.data_loader import CriteoDataLoader
from src.causal_engine import SingleModelSLearner, TwoModelTLearner, TrueDoublyRobustAIPW
from src.evaluate_metrics import compute_qini_curve

# 1. Load Criteo Benchmark Dataset
loader = CriteoDataLoader(data_dir="data", sample_size=100000, random_state=42)
df = loader.load_processed_data()

feature_cols = [f'f{i}' for i in range(12)]
X = df[feature_cols].values
T = df['treatment'].values
Y = df['conversion'].values

X_train, X_test, T_train, T_test, Y_train, Y_test = train_test_split(
    X, T, Y, test_size=0.25, random_state=42, stratify=T
)

print(f"Total Criteo Records Analyzed : {len(df):,}")
print(f"Training Partition Matrix     : {X_train.shape}")
print(f"Holdout Test Matrix           : {X_test.shape}")
print(f"Treatment Group Distribution  : {T_train.mean()*100:.1f}% Treated / {(1-T_train.mean())*100:.1f}% Control")

Loading verified benchmark dataset from data\criteo_uplift_processed.csv...
Total Criteo Records Analyzed : 50,000
Training Partition Matrix     : (37500, 12)
Holdout Test Matrix           : (12500, 12)
Treatment Group Distribution  : 84.9% Treated / 15.1% Control


## 2. Train Causal Meta-Learners & Doubly Robust AIPW

In [3]:
# 2. Train S-Learner, T-Learner, and Doubly Robust AIPW
models = {
    "Single-Model S-Learner": SingleModelSLearner(n_estimators=60, random_state=42),
    "Two-Model T-Learner": TwoModelTLearner(n_estimators=60, random_state=42),
    "True Doubly Robust AIPW (DML)": TrueDoublyRobustAIPW(n_folds=5, n_estimators=60, random_state=42)
}

predictions = {}
for name, model in models.items():
    print(f"Fitting {name} on {len(X_train):,} training records...")
    model.fit(X_train, T_train, Y_train)
    preds = model.predict_uplift(X_test)
    predictions[name] = preds
    print(f"-> {name} converged. Mean predicted uplift: {preds.mean():.6f}")

Fitting Single-Model S-Learner on 37,500 training records...
-> Single-Model S-Learner converged. Mean predicted uplift: 0.000183
Fitting Two-Model T-Learner on 37,500 training records...
-> Two-Model T-Learner converged. Mean predicted uplift: 0.000373
Fitting True Doubly Robust AIPW (DML) on 37,500 training records...
-> True Doubly Robust AIPW (DML) converged. Mean predicted uplift: 0.000343


## 3. Evaluate Qini Curves and Area Under Uplift Curve (AUUC)

In [5]:
print("=" * 85)
print("CAUSAL MODEL SELECTION TOURNAMENT RESULTS (QINI AUUC METRICS)")
print("=" * 85)

_, _, rand_auuc, _ = compute_qini_curve(Y_test, T_test, np.zeros(len(Y_test)))
print(f"{'Random Targeting Baseline':<35} | AUUC: {rand_auuc:>7.2f} | Lift:  +0.0%")

for name, preds in predictions.items():
    _, _, auuc, lift_ratio = compute_qini_curve(Y_test, T_test, preds)
    print(f"{name:<35} | AUUC: {auuc:>7.2f} | Lift: {lift_ratio*100:>+5.1f}%")
print("=" * 85)

CAUSAL MODEL SELECTION TOURNAMENT RESULTS (QINI AUUC METRICS)
Random Targeting Baseline           | AUUC:   -0.94 | Lift:  +0.0%
Single-Model S-Learner              | AUUC:   -2.64 | Lift: -221.3%
Two-Model T-Learner                 | AUUC:    2.03 | Lift:  -6.8%
True Doubly Robust AIPW (DML)       | AUUC:   -3.87 | Lift: -278.0%
